In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import os

# Configure data routines
import pandas as pd

JupyterDash.infer_jupyter_proxy_config()


# ---------------------------------------------------------
# Import CRUD module from Project One
# ---------------------------------------------------------

from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"

# Keep your actual aacuser password here in Codio.
password = "YOUR_PASSWORD_HERE"

# Connect to MongoDB using CRUD module
db = AnimalShelter(username, password)

# Retrieve all AAC records
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId so Dash can serialize the data
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)


# ---------------------------------------------------------
# Grazioso Salvare Logo
# ---------------------------------------------------------

# This tries to locate the logo file automatically.
logo_file = None

for filename in os.listdir("."):
    if "grazioso" in filename.lower() and filename.lower().endswith(
        (".png", ".jpg", ".jpeg")
    ):
        logo_file = filename
        break

if logo_file is not None:
    encoded_image = base64.b64encode(
        open(logo_file, "rb").read()
    ).decode()

    logo = html.Img(
        src="data:image/png;base64,{}".format(encoded_image),
        style={
            "height": "110px",
            "marginRight": "25px"
        }
    )

else:
    logo = html.Div(
        "Grazioso Salvare Logo",
        style={
            "fontWeight": "bold",
            "fontSize": "22px"
        }
    )


app.layout = html.Div([

    # Logo and unique identifier
    html.Div(
        [
            logo,

            html.Div([
                html.H1(
                    "Grazioso Salvare Rescue Dashboard"
                ),

                html.H3(
                    "Created by Roderick Scott - CS 340"
                )
            ])
        ],

        style={
            "display": "flex",
            "alignItems": "center",
            "justifyContent": "center"
        }
    ),

    html.Hr(),

    # -----------------------------------------------------
    # Interactive Filtering Options
    # -----------------------------------------------------

    html.H3("Rescue Type Filter"),

    dcc.RadioItems(
        id="filter-type",

        options=[
            {
                "label": "Reset",
                "value": "Reset"
            },
            {
                "label": "Water Rescue",
                "value": "Water"
            },
            {
                "label": "Mountain or Wilderness Rescue",
                "value": "Mountain"
            },
            {
                "label": "Disaster or Individual Tracking",
                "value": "Disaster"
            }
        ],

        value="Reset",

        inline=True,

        labelStyle={
            "marginRight": "25px"
        }
    ),

    html.Hr(),

    # -----------------------------------------------------
    # Interactive Data Table
    # -----------------------------------------------------

    dash_table.DataTable(
        id="datatable-id",

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict("records"),

        # Required for map selection
        row_selectable="single",

        # Automatically select first row
        selected_rows=[0],

        # User-friendly features
        page_size=10,

        page_action="native",

        sort_action="native",
        sort_mode="multi",

        filter_action="native",

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "220px",
            "whiteSpace": "normal"
        },

        style_header={
            "fontWeight": "bold"
        }
    ),

    html.Br(),
    html.Hr(),

    # -----------------------------------------------------
    # Charts
    # -----------------------------------------------------

    html.Div(
        className="row",

        style={
            "display": "flex"
        },

        children=[

            # Breed chart
            html.Div(
                id="graph-id",
                className="col s12 m6",

                style={
                    "width": "50%"
                }
            ),

            # Map
            html.Div(
                id="map-id",
                className="col s12 m6",

                style={
                    "width": "50%"
                }
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################


# ---------------------------------------------------------
# Rescue Type Filter
# ---------------------------------------------------------

@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "selected_rows")
    ],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    # Water Rescue
    if filter_type == "Water":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },

            "sex_upon_outcome": "Intact Female",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Mountain or Wilderness Rescue
    elif filter_type == "Mountain":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Disaster or Individual Tracking
    elif filter_type == "Disaster":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # Reset
    else:
        query = {}

    results = db.read(query)

    filtered_df = pd.DataFrame.from_records(results)

    if "_id" in filtered_df.columns:
        filtered_df.drop(
            columns=["_id"],
            inplace=True
        )

    # Re-select first row whenever filter changes
    selected = [0] if not filtered_df.empty else []

    return filtered_df.to_dict("records"), selected


# ---------------------------------------------------------
# Breed Distribution Chart
# ---------------------------------------------------------

@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return html.P(
            "No animals match the selected rescue type."
        )

    breed_counts = (
        dff["breed"]
        .fillna("Unknown")
        .value_counts()
        .head(10)
        .reset_index()
    )

    breed_counts.columns = [
        "breed",
        "count"
    ]

    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                names="breed",
                values="count",
                title="Breed Distribution"
            )
        )
    ]


# ---------------------------------------------------------
# Highlight Selected Table Columns
# ---------------------------------------------------------

@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional"
    ),

    [
        Input(
            "datatable-id",
            "selected_columns"
        )
    ]
)
def update_styles(selected_columns):

    if selected_columns is None:
        selected_columns = []

    return [
        {
            "if": {
                "column_id": i
            },

            "background_color":
                "#D2F3FF"
        }

        for i in selected_columns
    ]


# ---------------------------------------------------------
# Geolocation Map
# ---------------------------------------------------------

@app.callback(
    Output(
        "map-id",
        "children"
    ),

    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),

        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(viewData, index):

    if viewData is None:
        dff = df.copy()

    else:
        dff = pd.DataFrame.from_dict(
            viewData
        )

    if dff.empty:
        return html.P(
            "No location data available."
        )

    # Default to first row
    if index is None or len(index) == 0:
        row = 0

    else:
        row = index[0]

    # Make sure selected row still exists
    if row >= len(dff):
        row = 0

    latitude = dff.iloc[row][
        "location_lat"
    ]

    longitude = dff.iloc[row][
        "location_long"
    ]

    breed = dff.iloc[row][
        "breed"
    ]

    animal_name = dff.iloc[row][
        "name"
    ]

    return [

        dl.Map(

            style={
                "width": "100%",
                "height": "450px"
            },

            center=[
                latitude,
                longitude
            ],

            zoom=10,

            children=[

                dl.TileLayer(
                    id="base-layer-id"
                ),

                dl.Marker(

                    position=[
                        latitude,
                        longitude
                    ],

                    children=[

                        dl.Tooltip(
                            str(breed)
                        ),

                        dl.Popup([

                            html.H3(
                                "Animal Name"
                            ),

                            html.P(
                                str(animal_name)
                            )
                        ])
                    ]
                )
            ]
        )
    ]


# ---------------------------------------------------------
# Run App
# ---------------------------------------------------------

# Using 8052 avoids conflict with your previous dashboards.
app.run_server(port=8052)

Dash app running on https://alienpolitic-shockframe-3000.codio.io/proxy/8052/
